In [1]:
import numpy as np
import tensorflow as tf
import jax
import jax.numpy as jnp
import flax.linen as nn
import optax
from flax.training import train_state

import time

I0000 00:00:1776476542.066701   33933 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1776476542.067940   33933 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1776476542.118940   33933 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1776476543.498518   33933 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONE

In [2]:
print('Cores Found:', jax.device_count())

Cores Found: 4


In [3]:
with tf.io.gfile.GFile("gs://fi2010-lob-data/BenchmarkDatasets/NoAuction/1.NoAuction_Zscore/NoAuction_Zscore_Training/Train_Dst_NoAuction_ZScore_CF_1.txt", 'r') as f:
    raw_data = np.loadtxt(f)

In [4]:
X_train = raw_data[:40,:].T
y_train = raw_data[145,:].T - 1
y_train = jnp.array(y_train, dtype=jnp.int32)
print(f'Checking shape: {X_train.shape, y_train.shape}')

Checking shape: ((39512, 40), (39512,))


In [5]:
TIME_STEPS = 127
BATCH_SIZE = 32

dataset = tf.keras.utils.timeseries_dataset_from_array(
    data=X_train,
    targets=y_train[TIME_STEPS - 1:], 
    sequence_length=TIME_STEPS,
    sequence_stride=1,
    batch_size=BATCH_SIZE,
)
dataset = dataset.unbatch().batch(BATCH_SIZE, drop_remainder=True)

dataset = dataset.prefetch(tf.data.AUTOTUNE)

for batch_x, batch_y in dataset.take(1):
    print(f"Shape of X: {batch_x.shape}")
    print(f"Shape of Y: {batch_y.shape},{type(batch_y)}")

Shape of X: (32, 127, 40)
Shape of Y: (32,),<class 'tensorflow.python.framework.ops.EagerTensor'>


E0000 00:00:1776476556.352659   33933 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [6]:
class TCNBlock(nn.Module):
    features: int
    dilation: int
    kernel_size: int = 3

    @nn.compact 
    def __call__(self, x):
        y = nn.Conv(
            features=self.features,
            kernel_size=(self.kernel_size,),
            kernel_dilation=(self.dilation,),
            padding='CAUSAL',
            dtype = jnp.bfloat16
        )(x)
        return nn.relu(y)

class TCN(nn.Module):
    features: int = 64

    @nn.compact
    def __call__(self, x):
        d = [1, 2, 4, 8, 16, 32]
        
        for dilation in d:
            x = TCNBlock(
                features=self.features, 
                dilation=dilation, 
                kernel_size=3
            )(x)
            
        Output = x[:, -1, :]
        logits = nn.Dense(features=3, dtype = jnp.bfloat16)(Output)
        return logits

In [7]:
def create_train_state(rng, model, learning_rate = 0.01):
    dummy_x = jnp.ones((1, 100, 40)) 
    variables = model.init(rng, dummy_x)
    params = variables['params'] 

    tx = optax.adam(learning_rate)

    return train_state.TrainState.create(
        apply_fn=model.apply,
        params=params,
        tx=tx,
    )

In [8]:
@jax.jit
def train_step(state, batch_x, batch_y):
    # loss function
    def loss_fn(params):
        logits = state.apply_fn({'params': params}, batch_x)
        
        loss = optax.softmax_cross_entropy_with_integer_labels(
            logits=logits, labels=batch_y
        ).mean()
        
        preds = jnp.argmax(logits, axis=-1)
        accuracy = jnp.mean(preds == batch_y)
        
        return loss, accuracy 
    grad_fn = jax.value_and_grad(loss_fn, has_aux=True)
    
    (loss, accuracy), grads = grad_fn(state.params)

    state = state.apply_gradients(grads=grads)
    
    return state, loss, accuracy

In [9]:
# Create a jax state variable for calculation.
rng = jax.random.PRNGKey(42) 
tcn_model = TCN()
state = create_train_state(rng, tcn_model, learning_rate=1e-3)

In [10]:
EPOCHS = 3

print(f"--- Small sample test:  {EPOCHS} Epoch Training ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # Record the state of each epoch
    epoch_loss = 0.0
    epoch_accuracy = 0.0
    batch_count = 0
    
    for batch_x, batch_y in dataset:
        
        x_jax = jnp.array(batch_x.numpy(), dtype = jnp.bfloat16)
        y_jax = jnp.array(batch_y.numpy(), dtype = jnp.int32)
        
        state, loss, accuracy = train_step(state, x_jax, y_jax)
        
        epoch_loss += loss
        epoch_accuracy += accuracy
        batch_count += 1
        
        if batch_count == 1 and epoch == 0:
            print(f"✅ First Batch with initial Loss: {loss}, Accuracy: {accuracy}")

    avg_loss = epoch_loss / batch_count
    avg_acc = epoch_accuracy / batch_count
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Time: {epoch_time}s | Avg Loss: {avg_loss} | Avg Accuracy: {avg_acc}")

print("--- TCN Training test done. ---")

--- Small sample test:  3 Epoch Training ---
✅ First Batch with initial Loss: 1.11719, Accuracy: 0.25
Epoch 1/3 | Time: 3.251471996307373s | Avg Loss: 0.427734 | Avg Accuracy: 0.5206555128097534
Epoch 2/3 | Time: 1.8779523372650146s | Avg Loss: 0.425781 | Avg Accuracy: 0.5232723951339722
Epoch 3/3 | Time: 1.84812331199646s | Avg Loss: 0.427734 | Avg Accuracy: 0.5232723951339722
--- TCN Training test done. ---


In [14]:
num_devices = jax.local_device_count()
sharding = jax.sharding.PositionalSharding(jax.devices())

In [15]:
x_sharding = sharding.reshape(num_devices, 1, 1)

y_sharding = sharding.reshape(num_devices)

In [17]:
EPOCHS = 3
print(f"--- 🚀 Smart Sharding Test: {EPOCHS} Epoch Training ---")

for epoch in range(EPOCHS):
    start_time = time.time()
    
    # Record the state of each epoch
    epoch_loss = 0.0
    epoch_accuracy = 0.0
    batch_count = 0
    
    for batch_x, batch_y in dataset:
        
        # 💡 核心魔法：jax.device_put
        # 这一步会自动把 Numpy 数据按 x_sharding 的策略切片，并零拷贝送入 4 颗 TPU 的高带宽内存中！
        x_jax = jax.device_put(batch_x.numpy(), x_sharding).astype(jnp.bfloat16)
        y_jax = jax.device_put(batch_y.numpy(), y_sharding).astype(jnp.int32)
        
        # 像单机一样调用！XLA 编译器接管了一切跨核心通信
        state, loss, accuracy = train_step(state, x_jax, y_jax)
        
        # 取出标量值进行累加
        epoch_loss += loss.item()
        epoch_accuracy += accuracy.item()
        batch_count += 1
        
        if batch_count == 1 and epoch == 0:
            print(f"✅ First Batch with initial Loss: {loss.item():.4f}, Accuracy: {accuracy.item():.4f}")

    avg_loss = epoch_loss / batch_count
    avg_acc = epoch_accuracy / batch_count
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} | Time: {epoch_time:.2f}s | Avg Loss: {avg_loss:.4f} | Avg Accuracy: {avg_acc:.4f}")

print("--- 🎉 TCN Training test done. ---")

--- 🚀 Smart Sharding Test: 3 Epoch Training ---
✅ First Batch with initial Loss: 1.0391, Accuracy: 0.5000
Epoch 1/3 | Time: 4.07s | Avg Loss: 1.0046 | Avg Accuracy: 0.5232
Epoch 2/3 | Time: 3.15s | Avg Loss: 1.0057 | Avg Accuracy: 0.5233
Epoch 3/3 | Time: 3.13s | Avg Loss: 1.0001 | Avg Accuracy: 0.5233
--- 🎉 TCN Training test done. ---


# Test TCN_model.py

In [17]:
st = time.time()
%run TCN_model.py
print(time.time()-st)

Loss: [17.63721 17.63721 17.63721 17.63721], Accuracy: [0. 0. 0. 0.]
1.718247652053833


In [18]:
from TCN_model import TCN, init_train_state, train_step

In [21]:
rng = jax.random.PRNGKey(3721)
model = TCN()
state, dropout_rng = init_train_state(rng, model)

In [22]:
for epoch in range(EPOCHS):
    start_time = time.time()
    
    # 用于记录当前 Epoch 的累计状态
    epoch_loss = 0.0
    epoch_accuracy = 0.0
    batch_count = 0
    
    # 遍历我们之前做好的 1232 个 Batch 的 tf.data 管道
    for batch_x, batch_y in dataset:
        
        x_jax = jnp.array(batch_x.numpy(), dtype = jnp.bfloat16)
        y_jax = jnp.array(batch_y.numpy(), dtype = jnp.int32)
        
        # 核心爆炸输出：执行单步训练！
        # state 被送进去，伴随着 XLA 机器码的疯狂运算，吐出一个更新了权重的全新 state
        state, loss, accuracy = train_step(state, x_jax, y_jax, dropout_rng)
        
        # 累加指标
        epoch_loss += loss
        epoch_accuracy += accuracy
        batch_count += 1
        
        # 打印一下第一个 Batch 的进度，证明计算图已经打通并开始求导
        if batch_count == 1 and epoch == 0:
            print(f"✅ 第 1 个 Batch 前向/反向传播极速穿透！初始 Loss: {loss}, Accuracy: {accuracy}")

    # 计算整个 Epoch 的平均指标
    avg_loss = epoch_loss / batch_count
    avg_acc = epoch_accuracy / batch_count
    epoch_time = time.time() - start_time
    
    print(f"Epoch {epoch+1}/{EPOCHS} | 耗时: {epoch_time}s | 平均 Loss: {avg_loss} | 平均 Accuracy: {avg_acc}")

print("--- 🎉 恭喜！TCN 训练大循环彻底竣工！ ---")

✅ 第 1 个 Batch 前向/反向传播极速穿透！初始 Loss: 2.1083521842956543, Accuracy: 0.21875
Epoch 1/3 | 耗时: 4.1758575439453125s | 平均 Loss: 1.0702283382415771 | 平均 Accuracy: 0.49469006061553955
Epoch 2/3 | 耗时: 2.00984263420105s | 平均 Loss: 0.983942449092865 | 平均 Accuracy: 0.5278201699256897
Epoch 3/3 | 耗时: 2.0808401107788086s | 平均 Loss: 0.9600217938423157 | 平均 Accuracy: 0.5452998280525208
--- 🎉 恭喜！TCN 训练大循环彻底竣工！ ---
